In [1]:
import os
# Cài đặt pyvi để phân từ tiếng Việt (Bắt buộc cho PhoBERT để đạt điểm cao)
os.system('pip install pyvi')

import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from pyvi import ViTokenizer
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm
import random

# ==========================================
# 1. SET SEED & CONFIGURATION
# ==========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

class Config:
    model_name = "vinai/phobert-base"
    max_len = 120    
    batch_size = 32  
    epochs = 4       
    learning_rate = 2e-5 # Cột mốc LR cho BERT
    weight_decay = 0.01
    n_folds = 5
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    num_classes = 3  # Sentiment: 0, 1, 2
    num_topics = 4   # Topic: 0, 1, 2, 3
    
    warmup_ratio = 0.1
    topic_loss_weight = 0.3  
    
config = Config()
print(f"Using device: {config.device}")

# ==========================================
# 2. TIỀN XỬ LÝ & LOAD DỮ LIỆU
# ==========================================
def clean_text(text):
    text = str(text).lower()
    # Xóa các khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()
    # Các bước chuẩn hóa teencode cơ bản có thể thêm vào đây
    text = re.sub(r'\bko\b', 'không', text)
    text = re.sub(r'\bdc\b', 'được', text)
    text = re.sub(r'\br\b', 'rồi', text)
    return text

try:
    train_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/train.csv')
    test_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/test.csv')
except FileNotFoundError:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

print("Đang làm sạch và phân từ tiếng Việt (Word Segmentation)...")
train_df['sentence'] = train_df['sentence'].apply(lambda x: ViTokenizer.tokenize(clean_text(x)))
test_df['sentence'] = test_df['sentence'].apply(lambda x: ViTokenizer.tokenize(clean_text(x)))

# Tính toán Class Weights
sent_weights = compute_class_weight('balanced', classes=np.unique(train_df['sentiment']), y=train_df['sentiment'])
sent_weights = torch.tensor(sent_weights, dtype=torch.float).to(config.device)

topic_weights = compute_class_weight('balanced', classes=np.unique(train_df['topic']), y=train_df['topic'])
topic_weights = torch.tensor(topic_weights, dtype=torch.float).to(config.device)

# ==========================================
# 3. DATASET
# ==========================================
class JointSentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, is_train=True):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_train = is_train
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        sentence = str(self.df.iloc[idx]['sentence'])
        
        encoding = self.tokenizer(
            sentence,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        
        if self.is_train:
            item['sentiment'] = torch.tensor(self.df.iloc[idx]['sentiment'], dtype=torch.long)
            item['topic'] = torch.tensor(self.df.iloc[idx]['topic'], dtype=torch.long)
            
        return item

# ==========================================
# 4. MÔ HÌNH CẢI TIẾN (Mean Pooling & Multi-Sample Dropout)
# ==========================================
class JointPhoBERT_Improved(nn.Module):
    def __init__(self, model_name, num_topics, num_sentiments):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        
        # Multi-Sample Dropout (5 dropouts với tỷ lệ 0.2)
        self.dropouts = nn.ModuleList([nn.Dropout(0.2) for _ in range(5)]) 
        
        self.topic_classifier = nn.Linear(self.bert.config.hidden_size, num_topics)
        self.layer_norm = nn.LayerNorm(self.bert.config.hidden_size + num_topics)
        
        self.sentiment_classifier = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size + num_topics, 128),
            nn.GELU(),
            nn.Linear(128, num_sentiments)
        )
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state
        
        # Mean Pooling
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        pooled_output = sum_embeddings / sum_mask
        
        # Dự đoán Topic với Multi-Sample Dropout
        topic_logits = sum(self.topic_classifier(dropout(pooled_output)) for dropout in self.dropouts) / len(self.dropouts)
        topic_probs = torch.softmax(topic_logits, dim=1)
        
        # Nối feature và chuẩn hóa
        combined_features = torch.cat((pooled_output, topic_probs), dim=1)
        combined_features = self.layer_norm(combined_features)
        
        # Dự đoán Sentiment với Multi-Sample Dropout
        sentiment_logits = sum(self.sentiment_classifier(dropout(combined_features)) for dropout in self.dropouts) / len(self.dropouts)
        
        return topic_logits, sentiment_logits

# ==========================================
# 5. TRAINING LOOP
# ==========================================
def train_epoch(model, data_loader, optimizer, scheduler, crit_topic, crit_sent, device, scaler):
    model.train()
    losses = []
    all_preds = []
    all_labels = []
    
    progress_bar = tqdm(data_loader, desc='Training', leave=False)
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_sent = batch['sentiment'].to(device)
        labels_topic = batch['topic'].to(device)
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            topic_logits, sentiment_logits = model(input_ids, attention_mask)
            loss_topic = crit_topic(topic_logits, labels_topic)
            loss_sent = crit_sent(sentiment_logits, labels_sent)
            loss = loss_sent + config.topic_loss_weight * loss_topic
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        losses.append(loss.item())
        _, preds_sent = torch.max(sentiment_logits, dim=1)
        all_preds.extend(preds_sent.cpu().numpy())
        all_labels.extend(labels_sent.cpu().numpy())
        progress_bar.set_postfix({'loss': np.mean(losses)})
    
    return np.mean(losses), accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds, average='weighted')

def eval_epoch(model, data_loader, crit_topic, crit_sent, device):
    model.eval()
    losses = []
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        progress_bar = tqdm(data_loader, desc='Evaluating', leave=False)
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels_sent = batch['sentiment'].to(device)
            labels_topic = batch['topic'].to(device)
            
            with torch.cuda.amp.autocast():
                topic_logits, sentiment_logits = model(input_ids, attention_mask)
                loss_topic = crit_topic(topic_logits, labels_topic)
                loss_sent = crit_sent(sentiment_logits, labels_sent)
                loss = loss_sent + config.topic_loss_weight * loss_topic
            
            losses.append(loss.item())
            _, preds_sent = torch.max(sentiment_logits, dim=1)
            all_preds.extend(preds_sent.cpu().numpy())
            all_labels.extend(labels_sent.cpu().numpy())
            
    return np.mean(losses), accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds, average='weighted')

# Cập nhật hàm predict để trả về xác suất (Soft Voting)
def predict_proba(model, data_loader, device):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc='Predicting Proba'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            with torch.cuda.amp.autocast():
                _, sentiment_logits = model(input_ids, attention_mask)
            probs = torch.softmax(sentiment_logits, dim=1)
            all_probs.extend(probs.cpu().numpy())
    return np.array(all_probs)

# ==========================================
# 6. MAIN EXECUTION (K-FOLD)
# ==========================================
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=False)

test_dataset = JointSentimentDataset(test_df, tokenizer, config.max_len, is_train=False)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size*2, shuffle=False) 

skf = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=42)
X = train_df['sentence'].values
y = train_df['sentiment'].values

fold_probabilities = []
val_scores = []

crit_topic = nn.CrossEntropyLoss(weight=topic_weights, label_smoothing=0.05)
crit_sent = nn.CrossEntropyLoss(weight=sent_weights, label_smoothing=0.05)
scaler = torch.cuda.amp.GradScaler()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n{'='*50}\nFold {fold + 1}/{config.n_folds}\n{'='*50}")
    
    train_fold = train_df.iloc[train_idx].reset_index(drop=True)
    val_fold = train_df.iloc[val_idx].reset_index(drop=True)
    
    train_dataset = JointSentimentDataset(train_fold, tokenizer, config.max_len, is_train=True)
    val_dataset = JointSentimentDataset(val_fold, tokenizer, config.max_len, is_train=True)
    
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)
    
    model = JointPhoBERT_Improved(config.model_name, config.num_topics, config.num_classes).to(config.device)
    
    # Discriminative Fine-Tuning: Learning Rate riêng cho BERT và Classifier
    bert_params = list(model.bert.parameters())
    head_params = list(model.topic_classifier.parameters()) + list(model.sentiment_classifier.parameters()) + list(model.layer_norm.parameters())
    
    optimizer_grouped_parameters = [
        {'params': bert_params, 'lr': config.learning_rate, 'weight_decay': config.weight_decay},
        {'params': head_params, 'lr': config.learning_rate * 10, 'weight_decay': config.weight_decay}
    ]
    optimizer = AdamW(optimizer_grouped_parameters)
    
    total_steps = len(train_loader) * config.epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * config.warmup_ratio), num_training_steps=total_steps)
    
    best_val_f1 = 0
    best_model_state = None
    
    for epoch in range(config.epochs):
        print(f"Epoch {epoch + 1}/{config.epochs}")
        train_loss, train_acc, train_f1 = train_epoch(model, train_loader, optimizer, scheduler, crit_topic, crit_sent, config.device, scaler)
        print(f"Train - Loss: {train_loss:.4f}, F1: {train_f1:.4f}")
        
        val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, crit_topic, crit_sent, config.device)
        print(f"Val   - Loss: {val_loss:.4f}, F1: {val_f1:.4f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"--> Mới tạo kỷ lục F1: {val_f1:.4f}")
            
    model.load_state_dict(best_model_state)
    model = model.to(config.device)
    val_scores.append(best_val_f1)
    
    # Dự đoán xác suất cho Test set (Soft Voting)
    fold_probs = predict_proba(model, test_loader, config.device)
    fold_probabilities.append(fold_probs)

print(f"\nCross-validation scores: {val_scores}")
print(f"Mean CV F1: {np.mean(val_scores):.4f} (+/- {np.std(val_scores):.4f})")

# ==========================================
# 7. SOFT VOTING ENSEMBLE SUBMISSION
# ==========================================
fold_probabilities = np.array(fold_probabilities)

# Lấy trung bình cộng xác suất của các Folds
mean_probabilities = np.mean(fold_probabilities, axis=0)

# Chọn nhãn có xác suất cao nhất
final_predictions = np.argmax(mean_probabilities, axis=1)

submission = pd.DataFrame({'id': test_df['id'], 'sentiment': final_predictions})
submission.to_csv('/kaggle/working/submission_improved.csv', index=False)
print("\nĐã lưu file: /kaggle/working/submission_improved.csv")
print(submission['sentiment'].value_counts().sort_index())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.7 MB/s eta 0:00:00
Using device: cuda
Đang làm sạch và phân từ tiếng Việt (Word Segmentation)...
Loading tokenizer...


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Val   - Loss: 0.9380, F1: 0.9372
--> Mới tạo kỷ lục F1: 0.9372
Epoch 4/4


Train - Loss: 0.7622, F1: 0.9658


Val   - Loss: 0.9407, F1: 0.9391
--> Mới tạo kỷ lục F1: 0.9391


Predicting Proba: 100%|██████████| 76/76 [00:07<00:00,  9.56it/s]



Cross-validation scores: [0.9474144400362025, 0.942785449800732, 0.936700679526924, 0.9420617060554123, 0.9390573830871742]
Mean CV F1: 0.9416 (+/- 0.0036)

Đã lưu file: /kaggle/working/submission_improved.csv
sentiment
0    2204
1     217
2    2432
Name: count, dtype: int64
